## chains-> The Output of one component become input of second componenet ##

**Sequential Chain**

In [6]:
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

hug_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hug_token,
    max_new_tokens=300
)

model = ChatHuggingFace(llm=llm)
parser = StrOutputParser()

temp1 = ChatPromptTemplate.from_template("""Write down detail explanation on : {topic}""")
temp2 = ChatPromptTemplate.from_template("""Write down 5 points summary of the : {text}""")

chain = temp1 | model | parser | temp2 | model | parser
result = chain.invoke({"topic":"poverty in Pakistan"})
print(result)

Sure, here are five key points summarizing poverty in Pakistan:

1. **Definition and Measurement**:
   - Poverty in Pakistan is officially defined by the government as an income level below a certain threshold.
   - The World Bank measures poverty using consumption per capita based on purchasing power parity (PPP).

2. **Poverty Rate and Trends**:
   - As of the latest data (2020-2021), the poverty rate is around 27.4%, with approximately 47 million people living in poverty.
   - This rate has been relatively stable since around 2010, where it was around 23%.

3. **Poverty Line**:
   - The poverty line is set at approximately $1.90 per person per day (PPP).
   - Households are considered poor if their daily per capita consumption is below this threshold.

4. **Causes of Poverty**:
   - **Economic Factors**: Economic instability, including volatile growth rates and high inflation, and persistent trade deficits contribute to poverty.
   - **Inefficient Distribution**: There is a signific

**Parallel Chains**

In [7]:
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel


load_dotenv()

hug_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hug_token,
    max_new_tokens=300
)

model1 = ChatHuggingFace(llm=llm)
parser = StrOutputParser()

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

model2 = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [9]:
prompt1 = ChatPromptTemplate.from_template("""Make proper notes for this :{topic}""")
prompt2 = ChatPromptTemplate.from_template("""Make a Quiz notes for this :{topic}""")
prompt3 = ChatPromptTemplate.from_template("""Merge the given {notes} and {quiz} and make 1 document.""")


parallel_chain = RunnableParallel({
    "notes": prompt1 | model | parser,
    "quiz": prompt2 | model | parser
})

sequence_chain = prompt3 | model1 | parser

merge_chain = parallel_chain | sequence_chain

result = merge_chain.invoke({"topic":"Scikit Learn"})
print(result)

Great! Here's a structured quiz based on Scikit-learn to test your knowledge on the library. You can use this quiz to assess yourself or others on the key concepts and functions provided by Scikit-learn.

### Scikit-Learn Quiz

#### Part 1: Overview and Features

1. **What is Scikit-Learn primarily used for?**
   - A. Web development
   - B. Machine learning and data analysis
   - C. Game development
   - D. Database management

2. **Which Python libraries does Scikit-Learn depend on?**
   - A. NumPy, SciPy, and Pandas
   - B. TensorFlow, PyTorch, and Keras
   - C. NumPy, SciPy, and Matplotlib
   - D. Matplotlib, Plotly, and Seaborn

3. **What key feature of Scikit-Learn is highlighted by its modular design?**
   - A. Speed
   - B. Easy integration with other Python libraries
   - C. Memory efficiency
   - D. Scalability

4. **Which of the following is a common feature of Scikit-Learn's API?**
   - A. Inconsistent method names
   - B. Consistent method names across algorithms
   - C. C

**Conditional Workflows**

In [17]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from typing_extensions import Literal


class SentimentCheck(BaseModel):
  sentiment: Literal["positive", "negative"] = Field(
      description="The sentiment of the sentence"
  )


parser2 = PydanticOutputParser(pydantic_object=SentimentCheck)

prompt1 = ChatPromptTemplate.from_template(
    """What is the sentiment of this : {feedback}\n {format_instructions}""",
    partial_variables={
        "format_instructions": parser2.get_format_instructions()
    },
)

# Keep the feedback and extract only the sentiment string
classifier_chain = {
    "feedback": lambda x: x["feedback"],
    "sentiment": prompt1 | model1 | parser2 | (lambda x: x.sentiment),
}

prompt2 = ChatPromptTemplate.from_template(
    "write an appropriate response to this positive : {feedback}"
)
prompt3 = ChatPromptTemplate.from_template(
    "write an appropriate response to this negative : {feedback}"
)

branch_chain = RunnableBranch(
    (lambda x: x["sentiment"] == "positive", prompt2 | model1),
    (lambda x: x["sentiment"] == "negative", prompt3 | model1),
    RunnableLambda(lambda x: "not found the sentiment"),
)

chain = classifier_chain | branch_chain
print(chain.invoke({"feedback": "The Quality of food is too bad"}))

content="I'm sorry to hear that you didn't enjoy the food quality. We strive to provide high-quality meals, and it sounds like there may have been an issue. Could you please let me know more details about your experience? Are there any specific areas where you felt the food quality was subpar? This feedback will help us address the issue and improve our service. Thank you for bringing this to our attention." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 44, 'total_tokens': 127}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ff083-a86e-7b10-a82c-cc07c335fb74-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 44, 'output_tokens': 83, 'total_tokens': 127}
